In [2]:
import os
import numpy as np
import librosa
from glob import glob
from sklearn.metrics import roc_auc_score
from tensorflow.keras import layers, models

In [3]:
BASE_DIR = "processed_data"
SR = 16000
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512

WINDOW_FRAMES = 256
WINDOW_HOP = 128

EPOCHS = 30
BATCH_SIZE = 16

In [4]:
def extract_log_mel(y):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )
    return librosa.power_to_db(mel)


def sliding_windows(mel):
    windows = []
    for start in range(0, mel.shape[1] - WINDOW_FRAMES + 1, WINDOW_HOP):
        windows.append(mel[:, start:start + WINDOW_FRAMES])
    return windows

In [5]:
def build_autoencoder():
    model = models.Sequential([
        layers.Input(shape=(N_MELS, WINDOW_FRAMES, 1)),

        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(2),

        layers.Conv2D(16, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(2),

        layers.Conv2D(8, 3, activation="relu", padding="same"),

        layers.UpSampling2D(2),
        layers.Conv2D(16, 3, activation="relu", padding="same"),

        layers.UpSampling2D(2),
        layers.Conv2D(1, 3, activation="linear", padding="same")
    ])

    model.compile(optimizer="adam", loss="mse")
    return model

In [6]:
def load_train_windows(id_dir):
    files = glob(os.path.join(id_dir, "train", "normal", "*.wav"))
    X = []

    for f in files:
        y, _ = librosa.load(f, sr=SR)
        mel = extract_log_mel(y)

        if mel.shape[1] < WINDOW_FRAMES:
            mel = np.pad(mel, ((0, 0), (0, WINDOW_FRAMES - mel.shape[1])))

        windows = sliding_windows(mel)
        X.extend(windows)

    X = np.array(X)[..., np.newaxis]
    return X

In [7]:
def anomaly_score(model, path):
    y, _ = librosa.load(path, sr=SR)
    mel = extract_log_mel(y)

    if mel.shape[1] < WINDOW_FRAMES:
        mel = np.pad(mel, ((0, 0), (0, WINDOW_FRAMES - mel.shape[1])))

    windows = sliding_windows(mel)
    scores = []

    for w in windows:
        w = w[np.newaxis, ..., np.newaxis]
        recon = model.predict(w, verbose=0)
        scores.append(np.mean((w - recon) ** 2))

    return np.max(scores)

In [8]:
aucs = []

for machine_id in sorted(os.listdir(BASE_DIR)):
    id_dir = os.path.join(BASE_DIR, machine_id)
    if not os.path.isdir(id_dir):
        continue

    print(f"\nTraining model for {machine_id}")

    X_train = load_train_windows(id_dir)

    model = build_autoencoder()
    model.fit(
        X_train,
        X_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        verbose=0
    )

    normal_files = glob(os.path.join(id_dir, "test", "normal", "*.wav"))
    abnormal_files = glob(os.path.join(id_dir, "test", "abnormal", "*.wav"))

    scores_normal = [anomaly_score(model, f) for f in normal_files]
    scores_abnormal = [anomaly_score(model, f) for f in abnormal_files]

    y_true = np.concatenate([
        np.zeros(len(scores_normal)),
        np.ones(len(scores_abnormal))
    ])
    scores = np.concatenate([scores_normal, scores_abnormal])

    auc = roc_auc_score(y_true, scores)
    aucs.append(auc)

    print(f"{machine_id} ROC-AUC: {auc:.4f}")

print(f"\n Mean ROC-AUC across IDs: {np.mean(aucs):.4f}")


Training model for id_00
id_00 ROC-AUC: 0.5191

Training model for id_02
id_02 ROC-AUC: 0.5814

Training model for id_04
id_04 ROC-AUC: 0.5370

Training model for id_06
id_06 ROC-AUC: 0.8564

 Mean ROC-AUC across IDs: 0.6235
